In [1]:
# conda env 'at3d_pure_k9'
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.Retrieval import Retrieval

In [3]:
def setup_masks(r):
    brf_ths = {
        'DF': 0.120,
        'CF': 0.076,
        'BF': 0.054,
        'AF': 0.044,
        'AN': 0.040,
        'AA': 0.045,
        'BA': 0.052,
        'CA': 0.068,
        'DA': 0.092
    }
    r.add_threshold_mask(r.obs, 'MISR', 'threshold_mask', 'brf', brf_ths)
    # r.add_threshold_mask(r.sensors_retrieval, 'MISR', 'tau_mask', 'optical_depth', 0.3)
    r.add_ray_containment_mask(mask_name='ray_containment')

def misr_S_1(y):
    return np.diag(1 / (0.04 * y)**2)

def get_stats(r_dir, category, name):
    r = Retrieval(r_dir)
    r.load_retrieval(extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])
    setup_masks(r)

    cloudy_mask_conds = {
        'threshold_mask': (1,1),
        'ray_containment': (2, 2)
    }
    all_mask_conds = {
        'ray_containment': (2, 2)
    }

    r_stats = {}
    r_stats['Category'] = category
    r_stats['Name'] = name
    r_stats['chi2 cloudy'] = r.get_chi2(misr_S_1, variable='I', mask_conds=cloudy_mask_conds)
    r_stats['chi2 all'] = r.get_chi2(misr_S_1, variable='I', mask_conds=all_mask_conds)
    r_stats['Mean CTH (km)'] = np.nanmean(r.get_cth(0.67, 'cloud'))
    r_stats['Std CTH (km)'] = np.nanstd(r.get_cth(0.67, 'cloud'))
    r_stats['Mean CBH (km)'] = np.nanmean(r.get_cbh(0.67, 'cloud'))
    r_stats['Std CBH (km)'] = np.nanstd(r.get_cbh(0.67, 'cloud'))
    r_stats['Mean CGT (km)'] = np.nanmean(r.get_geo_thickness(0.67, 'cloud'))
    r_stats['Std CGT (km)'] = np.nanstd(r.get_geo_thickness(0.67, 'cloud'))
    r_stats['CF'] = r.calc_CF(0.67, 'cloud')
    r_stats['Volume (km^3)'] = r.calc_volume(0.67, 'cloud')
    r_stats['Mean Ext (km^-1)'] = np.nanmean(r.get_extinction(0.67, 'cloud'))
    r_stats['Std Ext (km^-1)'] = np.nanstd(r.get_extinction(0.67, 'cloud'))

    return r_stats

# Surface and Aerosols

In [ ]:
v_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V10'

s_surf_s_d = get_stats(v_dir / 'default', 'Surface', '0.67 m/s (default)')
s_surf_f = get_stats(v_dir / 'surf_wind' / 'surf4p0', 'Surface', '4.0 m/s')

s_aod_l = get_stats(v_dir / 'aerosol' / 'aod0p150', 'Aerosol', '0.15')
s_aod_m_d = get_stats(v_dir / 'default', 'Aerosol', '0.20 (default)')
s_aod_h = get_stats(v_dir / 'aerosol' / 'aod0p250', 'Aerosol', '0.25')

s_amix21_d = get_stats(v_dir / 'default', 'Aerosol', 'Mix 21 (default)')
s_amix22 = get_stats(v_dir / 'aerosol' / 'mix22', 'Aerosol', 'Mix 22')
s_amix54 = get_stats(v_dir / 'aerosol' / 'mix54', 'Aerosol', 'Mix 54')

surf_aer_df = pd.DataFrame([
    s_surf_s_d,
    s_surf_f,
    s_aod_l,
    s_aod_m_d,
    s_aod_h,
    s_amix21_d,
    s_amix22,
    s_amix54
])

In [5]:
surf_aer_df

,Category,Name,chi2 cloudy,chi2 all,Mean CTH (km),Std CTH (km),Mean CBH (km),Std CBH (km),Mean CGT (km),Std CGT (km),CF,Volume (km^3),Mean Ext (km^-1),Std Ext (km^-1)
0,Surface,0.67 m/s (default),10.914991,19.047968,0.700996,0.164957,0.524040,0.144819,0.176956,0.175314,0.1663905325443787,13.710812500000001,7.969626,8.162009
1,Surface,4.0 m/s,8.244615,9.116992,0.670667,0.155216,0.461333,0.106503,0.209333,0.160797,0.2485207100591716,23.353000000000005,6.652439,7.024357
2,Aerosol,0.15,8.821809,8.740421,0.691309,0.157773,0.490773,0.128880,0.200536,0.168144,0.22059171597633137,19.511250000000008,6.224222,6.487624
3,Aerosol,0.20 (default),10.914991,19.047968,0.700996,0.164957,0.524040,0.144819,0.176956,0.175314,0.1663905325443787,13.710812500000001,7.969626,8.162009
4,Aerosol,0.25,17.272412,39.235901,0.696886,0.160850,0.559862,0.156899,0.137024,0.162419,0.13680473372781066,9.438000000000006,10.403011,9.993058
5,Aerosol,Mix 21 (default),10.914991,19.047968,0.700996,0.164957,0.524040,0.144819,0.176956,0.175314,0.1663905325443787,13.710812500000001,7.969626,8.162009
6,Aerosol,Mix 22,10.364889,18.137649,0.701122,0.163496,0.520477,0.144759,0.180645,0.172649,0.16875739644970414,14.2855625,7.609094,7.644280
7,Mix 54,Mix 54,11.464979,21.750962,0.688800,0.157590,0.533440,0.147790,0.155360,0.159823,0.14792899408284024,10.716062500000003,9.535604,10.597262


In [16]:
surf_aer_df.set_index(['Category', 'Name']).T

Category                     Surface                                  Aerosol  \
Name              0.67 m/s (default)             4.0 m/s                 0.15   
chi2 cloudy                10.914991            8.244615             8.821809   
chi2 all                   19.047968            9.116992             8.740421   
Mean CTH (km)               0.700996            0.670667             0.691309   
Std CTH (km)                0.164957            0.155216             0.157773   
Mean CBH (km)                0.52404            0.461333             0.490773   
Std CBH (km)                0.144819            0.106503              0.12888   
Mean CGT (km)               0.176956            0.209333             0.200536   
Std CGT (km)                0.175314            0.160797             0.168144   
CF                0.1663905325443787  0.2485207100591716  0.22059171597633137   
Volume (km^3)     13.710812500000001  23.353000000000005   19.511250000000008   
Mean Ext (km^-1)            7.969626            6.652439             6.224222   
Std Ext (km^-1)             8.162009            7.024357             6.487624   

Category                                                                       \
Name                  0.20 (default)                 0.25    Mix 21 (default)   
chi2 cloudy                10.914991            17.272412           10.914991   
chi2 all                   19.047968            39.235901           19.047968   
Mean CTH (km)               0.700996             0.696886            0.700996   
Std CTH (km)                0.164957              0.16085            0.164957   
Mean CBH (km)                0.52404             0.559862             0.52404   
Std CBH (km)                0.144819             0.156899            0.144819   
Mean CGT (km)               0.176956             0.137024            0.176956   
Std CGT (km)                0.175314             0.162419            0.175314   
CF                0.1663905325443787  0.13680473372781066  0.1663905325443787   
Volume (km^3)     13.710812500000001    9.438000000000006  13.710812500000001   
Mean Ext (km^-1)            7.969626            10.403011            7.969626   
Std Ext (km^-1)             8.162009             9.993058            8.162009   

Category                                            Mix 54  
Name                           Mix 22               Mix 54  
chi2 cloudy                 10.364889            11.464979  
chi2 all                    18.137649            21.750962  
Mean CTH (km)                0.701122               0.6888  
Std CTH (km)                 0.163496              0.15759  
Mean CBH (km)                0.520477              0.53344  
Std CBH (km)                 0.144759              0.14779  
Mean CGT (km)                0.180645              0.15536  
Std CGT (km)                 0.172649             0.159823  
CF                0.16875739644970414  0.14792899408284024  
Volume (km^3)              14.2855625   10.716062500000003  
Mean Ext (km^-1)             7.609094             9.535604  
Std Ext (km^-1)               7.64428            10.597262

# Horizontal Boundary Condition

In [12]:
v_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'

s_hbc_o_d = get_stats(v_dir / 'default', 'HBC', 'open (default)')
s_hbc_p = get_stats(v_dir / 'HBC' / 'periodic', 'HBC', 'periodic')

hbc_df = pd.DataFrame([
    s_hbc_o_d,
    s_hbc_p
])

In [13]:
hbc_df

,Category,Name,chi2 cloudy,chi2 all,Mean CTH (km),Std CTH (km),Mean CBH (km),Std CBH (km),Mean CGT (km),Std CGT (km),CF,Volume (km^3),Mean Ext (km^-1),Std Ext (km^-1)
0,HBC,open (default),10.795046,18.945354,0.691384,0.165095,0.527825,0.146339,0.163559,0.167736,0.16757396449704143,13.113375000000001,8.415283,8.583450
1,HBC,periodic,20.778908,33.931217,0.721157,0.155351,0.544292,0.143032,0.176865,0.177022,0.15550295857988167,12.534843750000006,5.432983,5.289284


In [17]:
hbc_df.set_index(['Category', 'Name']).T

Category                          HBC                     
Name                   open (default)             periodic
chi2 cloudy                 10.795046            20.778908
chi2 all                    18.945354            33.931217
Mean CTH (km)                0.691384             0.721157
Std CTH (km)                 0.165095             0.155351
Mean CBH (km)                0.527825             0.544292
Std CBH (km)                 0.146339             0.143032
Mean CGT (km)                0.163559             0.176865
Std CGT (km)                 0.167736             0.177022
CF                0.16757396449704143  0.15550295857988167
Volume (km^3)      13.113375000000001   12.534843750000006
Mean Ext (km^-1)             8.415283             5.432983
Std Ext (km^-1)               8.58345             5.289284

# Cloud Volume Mask

In [8]:
v_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'

s_cvm_lcs_d = get_stats(v_dir / 'default', 'CVM', 'LCL + CTH + SC (default)')
s_cvm_lc = get_stats(v_dir / 'CVM' / 'LCL_CTH', 'CVM', 'LCL + CTH')
s_cvm_unc = get_stats(v_dir / 'CVM' / 'unconstrained', 'CVM', 'Unconstrained')

cvm_df = pd.DataFrame([
    s_cvm_lcs_d,
    s_cvm_lc,
    s_cvm_unc
])

In [9]:
cvm_df

,Category,Name,chi2 cloudy,chi2 all,Mean CTH (km),Std CTH (km),Mean CBH (km),Std CBH (km),Mean CGT (km),Std CGT (km),CF,Volume (km^3),Mean Ext (km^-1),Std Ext (km^-1)
0,CVM,LCL + CTH + SC (default),10.795046,18.945354,0.691384,0.165095,0.527825,0.146339,0.163559,0.167736,0.16757396449704143,13.113375000000001,8.415283,8.583450
1,CVM,LCL + CTH,10.871575,18.990097,0.692928,0.165242,0.524894,0.143811,0.168034,0.170271,0.16733727810650886,13.211687500000004,8.362713,8.510878
2,CVM,Unconstrained,12.489859,20.644587,1.069487,0.345583,0.587692,0.369792,0.481795,0.414236,0.18461538461538463,20.588906250000008,6.040523,6.645118


In [18]:
cvm_df.set_index(['Category', 'Name']).T

Category                              CVM                       \
Name             LCL + CTH + SC (default)            LCL + CTH   
chi2 cloudy                     10.795046            10.871575   
chi2 all                        18.945354            18.990097   
Mean CTH (km)                    0.691384             0.692928   
Std CTH (km)                     0.165095             0.165242   
Mean CBH (km)                    0.527825             0.524894   
Std CBH (km)                     0.146339             0.143811   
Mean CGT (km)                    0.163559             0.168034   
Std CGT (km)                     0.167736             0.170271   
CF                    0.16757396449704143  0.16733727810650886   
Volume (km^3)          13.113375000000001   13.211687500000004   
Mean Ext (km^-1)                 8.415283             8.362713   
Std Ext (km^-1)                   8.58345             8.510878   

Category                               
Name                    Unconstrained  
chi2 cloudy                 12.489859  
chi2 all                    20.644587  
Mean CTH (km)                1.069487  
Std CTH (km)                 0.345583  
Mean CBH (km)                0.587692  
Std CBH (km)                 0.369792  
Mean CGT (km)                0.481795  
Std CGT (km)                 0.414236  
CF                0.18461538461538463  
Volume (km^3)      20.588906250000008  
Mean Ext (km^-1)             6.040523  
Std Ext (km^-1)              6.645118

# Cloud Motion

In [10]:
v_dir = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08'

s_cm_s = get_stats(v_dir / 'cloud_motion' / 'slow_unconstrained', 'Motion', '-0.35 m/s')
s_cm_d = get_stats(v_dir / 'CVM' / 'unconstrained', 'Motion', '1.5 m/s (default)')
s_cm_f = get_stats(v_dir / 'cloud_motion' / 'fast_unconstrained', 'Motion', '3.35 m/s')

motion_df = pd.DataFrame([
    s_cm_s,
    s_cm_d,
    s_cm_f
])

In [11]:
motion_df

,Category,Name,chi2 cloudy,chi2 all,Mean CTH (km),Std CTH (km),Mean CBH (km),Std CBH (km),Mean CGT (km),Std CGT (km),CF,Volume (km^3),Mean Ext (km^-1),Std Ext (km^-1)
0,Motion,-0.35 m/s,12.925494,20.948317,1.179383,0.286756,0.699877,0.386133,0.479506,0.419082,0.19171597633136095,20.638062500000007,6.538091,6.964304
1,Motion,1.5 m/s (default),12.489859,20.644587,1.069487,0.345583,0.587692,0.369792,0.481795,0.414236,0.18461538461538463,20.588906250000008,6.040523,6.645118
2,Motion,3.35 m/s,16.302323,23.634758,1.033990,0.385485,0.526724,0.408093,0.507266,0.445872,0.19218934911242602,20.967031250000005,5.663078,5.817712


In [15]:
motion_df.set_index(['Category', 'Name']).T

Category                       Motion                       \
Name                        -0.35 m/s    1.5 m/s (default)   
chi2 cloudy                 12.925494            12.489859   
chi2 all                    20.948317            20.644587   
Mean CTH (km)                1.179383             1.069487   
Std CTH (km)                 0.286756             0.345583   
Mean CBH (km)                0.699877             0.587692   
Std CBH (km)                 0.386133             0.369792   
Mean CGT (km)                0.479506             0.481795   
Std CGT (km)                 0.419082             0.414236   
CF                0.19171597633136095  0.18461538461538463   
Volume (km^3)      20.638062500000007   20.588906250000008   
Mean Ext (km^-1)             6.538091             6.040523   
Std Ext (km^-1)              6.964304             6.645118   

Category                               
Name                         3.35 m/s  
chi2 cloudy                 16.302323  
chi2 all                    23.634758  
Mean CTH (km)                 1.03399  
Std CTH (km)                 0.385485  
Mean CBH (km)                0.526724  
Std CBH (km)                 0.408093  
Mean CGT (km)                0.507266  
Std CGT (km)                 0.445872  
CF                0.19218934911242602  
Volume (km^3)      20.967031250000005  
Mean Ext (km^-1)             5.663078  
Std Ext (km^-1)              5.817712